##SET UP

In [1]:
import geopandas as gpd
import pandas as pd
import folium

In [37]:
import xyzservices.providers as xyz

In [4]:
from google.colab import drive
drive.mount('/drive')

Mounted at /drive


In [38]:
Carto_API = "cb1_4kgq_1_070c2d57cb01b4f1d14cac43"

In [16]:
DATA = "/drive/My Drive/595 Fall 2026 Problem Sets/PS9/MeghaJhawar/Data/"
counties = gpd.read_file(DATA + "NJ_Counties_NGJIN_2021.geojson").to_crs("EPSG:4326")

In [29]:
counties.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 23 columns):
 #   Column        Non-Null Count  Dtype   
---  ------        --------------  -----   
 0   OBJECTID      21 non-null     int32   
 1   COUNTY        21 non-null     object  
 2   COUNTY_LABEL  21 non-null     object  
 3   CO            21 non-null     object  
 4   GNIS_NAME     21 non-null     object  
 5   GNIS          21 non-null     object  
 6   FIPSSTCO      21 non-null     object  
 7   FIPSCO        21 non-null     object  
 8   ACRES         21 non-null     float64 
 9   SQ_MILES      21 non-null     float64 
 10  POP2010       21 non-null     int32   
 11  POP2000       21 non-null     int32   
 12  POP1990       21 non-null     int32   
 13  POP1980       21 non-null     int32   
 14  POPDEN2010    21 non-null     int32   
 15  POPDEN2000    21 non-null     int32   
 16  POPDEN1990    21 non-null     int32   
 17  POPDEN1980    21 non-null     int32   
 18  REGI

In [18]:
nj_state = counties.dissolve()

In [19]:
lihtc = pd.read_csv( DATA + "lihtc_family_geocoded.csv", low_memory=False)

In [20]:
lihtc.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 94 entries, 0 to 93
Data columns (total 44 columns):
 #   Column                                Non-Null Count  Dtype  
---  ------                                --------------  -----  
 0   Year                                  94 non-null     int64  
 1   round                                 11 non-null     object 
 2   Cycle                                 94 non-null     object 
 3   app_id                                94 non-null     object 
 4   Project                               94 non-null     object 
 5   Address                               94 non-null     object 
 6   Municipality                          94 non-null     object 
 7   County                                94 non-null     object 
 8   Developer                             94 non-null     object 
 9   Affordable Units                      94 non-null     int64  
 10  Total Units                           94 non-null     int64  
 11  LIHTC Type           

In [21]:
before_2018 = lihtc[lihtc["Year"]< 2018]
after_2018 = lihtc[lihtc["Year"]>=2018]

In [23]:
print("Before 2018", len(before_2018))
print("After 2018", len(after_2018))

Before 2018 28
After 2018 66


In [46]:
m = folium.Map(
    location=[40.1, -74.6],
    zoom_start=8,
    tiles=xyz.Esri.WorldTopoMap,
    attr=xyz.Esri.WorldTopoMap.html_attribution
)

folium.GeoJson(
    nj_state,
    name="NJ State Boundary",
    style_function=lambda x: {"color": "black", "weight": 3, "fill": False},
).add_to(m)

folium.GeoJson(
    counties[["COUNTY_LABEL", "geometry"]],
    name = "NJ Counties",
    style_function = lambda x: {"color": "black", "weight": 1, "fillOpacity": 0},
    tooltip = folium.GeoJsonTooltip(fields = ["COUNTY_LABEL"], aliases = ["County:"])
).add_to(m)

def add_points(data, layer_name, color):
  layer = folium.FeatureGroup(name=layer_name)
  for _, row in data.iterrows():
    popup_text = (
        f"<b>{row['Project']}</b><br>"
        f"{row['Address']}, {row['Municipality']}<br>"
        f"Year: {int(row['Year'])}<br>"
        f"Affordable units: {row['Affordable Units']}"
    )
    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=4,
        color=color,
        fill=True,
        fill_opacity = 0.7,
        popup = folium.Popup(popup_text, max_width = 250)
    ).add_to(layer)
  layer.add_to(m)

add_points(before_2018, "Before 2018", "#d62728")
add_points(after_2018, "After 2018", "#1f77b4")

folium.LayerControl(collapsed=False).add_to(m)

In [48]:
m.save("LIHTC_NJ.html")
print("Saved index.html")

Saved index.html
